# 03 — Baseline Experiments

Runs the classical inventory baselines, tunes their parameters by grid search, and locks the benchmark score that DQN (`04_dqn_training.ipynb`) and PPO (`05_ppo_training.ipynb`) must beat.

- **Reorder point (R, Q):** order `Q` units when inventory position is at or below `R`.
- **s, S:** when inventory position is at or below `s`, order up to `S`.
- **Random:** uniform random order quantity, as a floor.

**Protocol.** Parameters are chosen on tuning seeds `0–49`. The chosen parameters are scored once on held-out test seeds `1000–1099`, and that test score is the benchmark. All policies use the same seeds, so demand sequences are shared.

Costs are minimised. The RL reward is `-total_cost`, so a DQN or PPO policy beats the benchmark when its mean evaluation reward is above `-benchmark_cost`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the project packages importable when the notebook runs from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from baselines.benchmark import ENV_KWARGS, TABLES_DIR, TEST_SEEDS, TUNING_SEEDS, run_benchmark

print("Environment:", ENV_KWARGS)
print(f"Tuning seeds: {TUNING_SEEDS[0]}-{TUNING_SEEDS[-1]} ({len(TUNING_SEEDS)} episodes)")
print(f"Test seeds:   {TEST_SEEDS[0]}-{TEST_SEEDS[-1]} ({len(TEST_SEEDS)} episodes)")

## 1. Grid search and locked benchmark

This runs both grid searches on the tuning seeds, scores the winners on the test seeds, and writes the results to `results/tables/`. It takes about 20–30 seconds.

In [ ]:
benchmark = run_benchmark()
print("Written to:", TABLES_DIR)

## 2. Top grid-search results (tuning seeds)

In [ ]:
rp_grid = pd.read_csv(TABLES_DIR / "baseline_grid_reorder_point.csv")
ss_grid = pd.read_csv(TABLES_DIR / "baseline_grid_s_S.csv")

print("Reorder point (R, Q), lowest mean cost:")
display(rp_grid.nsmallest(10, "mean_cost")[["reorder_point", "order_quantity", "mean_cost", "stderr_cost"]])

print("s, S, lowest mean cost:")
display(ss_grid.nsmallest(10, "mean_cost")[["s", "S", "mean_cost", "stderr_cost"]])

## 3. Test-seed comparison and benchmark score

In [ ]:
rows = [
    {"policy": "reorder point", "params": benchmark["reorder_point"]["params"], **benchmark["reorder_point"]["test"]},
    {"policy": "s, S", "params": benchmark["s_S"]["params"], **benchmark["s_S"]["test"]},
    {"policy": "random", "params": {}, **benchmark["random"]["test"]},
]
comparison = pd.DataFrame(rows)[
    ["policy", "params", "mean_cost", "stderr_cost", "mean_stockout_rate", "mean_service_level"]
]
display(comparison)

target = benchmark["benchmark"]
print(f"Benchmark policy: {target['policy']}")
print(f"Benchmark cost (must beat, lower is better): {target['mean_cost']:.2f} ± {target['stderr_cost']:.2f}")
print(f"Benchmark reward (must beat, higher is better): {target['mean_reward']:.2f}")